# A2 — Text Feature Extraction

Continues from the dataset generator notebook. Loads the text labels and computes three text representations that will be used as prediction targets or inputs in the neural network notebook (A3).

**Prerequisite:** Run the dataset generator notebook first to create the `dataset/` folder.

**Representations:**
1. **One-Hot** — sentence-level identity vector (baseline)
2. **TF-IDF** — word + bigram weighted vectors
3. **SBERT** — dense 384-dim semantic embeddings

**CSV columns produced by the generator (used here):**
- `text` — natural language sentence
- `sym_num_dice` — number of dice (1–3)
- `sym_values` — space-separated die face values
- `sym_colours` — space-separated colour labels
- `sym_sizes` — space-separated size labels (small / medium / large)

## 1. Imports & Configuration

In [1]:
import os
import logging

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = logging.getLogger(__name__)

# Must match the generator notebook
DATASET_ROOT = "dataset"
SPLITS       = ["train", "val", "test"]

LABEL_PATHS = {
    split: os.path.join(DATASET_ROOT, split, "labels.csv")
    for split in SPLITS
}

# Symbol field config — must match generator notebook
VALID_COLOURS = ["white", "red", "blue", "green", "yellow", "purple", "peach"]
VALID_SIZES   = ["small", "medium", "large"]

# Verify files exist before proceeding
missing = [p for p in LABEL_PATHS.values() if not os.path.isfile(p)]
if missing:
    raise FileNotFoundError(
        f"Missing label files: {missing}\n"
        "Please run the dataset generator notebook first."
    )

print("All label files found. Ready to proceed.")

/opt/miniconda3/envs/msc/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All label files found. Ready to proceed.


## 2. Load Data

In [2]:
REQUIRED_COLUMNS = {"text", "sym_num_dice", "sym_values", "sym_colours", "sym_sizes", "image"}


def load_split(csv_path: str, split_name: str) -> pd.DataFrame:
    """Load a split CSV and validate expected columns and content."""
    df = pd.read_csv(csv_path)

    missing_cols = REQUIRED_COLUMNS - set(df.columns)
    if missing_cols:
        raise ValueError(
            f"[{split_name}] CSV missing columns: {missing_cols}.\n"
            "Ensure you ran the latest version of the generator notebook."
        )
    if df.empty:
        raise ValueError(f"[{split_name}] CSV is empty.")
    if df["text"].isnull().any():
        raise ValueError(f"[{split_name}] Found null values in 'text' column.")

    logger.info(f"Loaded {split_name}: {len(df)} samples.")
    return df


train_df = load_split(LABEL_PATHS["train"], "train")
val_df   = load_split(LABEL_PATHS["val"],   "val")
test_df  = load_split(LABEL_PATHS["test"],  "test")

train_texts = train_df["text"].tolist()
val_texts   = val_df["text"].tolist()
test_texts  = test_df["text"].tolist()

print(f"\nSplit sizes — Train: {len(train_texts)}, Val: {len(val_texts)}, Test: {len(test_texts)}")
print(f"\nSymbol columns present: {[c for c in train_df.columns if c.startswith('sym_')]}")
print(f"\nExample row:")
train_df.head(3)

INFO: Loaded train: 4181 samples.
INFO: Loaded val: 899 samples.
INFO: Loaded test: 898 samples.



Split sizes — Train: 4181, Val: 899, Test: 898

Symbol columns present: ['sym_num_dice', 'sym_values', 'sym_colours', 'sym_sizes']

Example row:


,image,text,sym_num_dice,sym_values,sym_colours,sym_sizes
0,sample_00000.png,The image shows a small purple die showing two...,3,2 1 2,purple green blue,small small medium
1,sample_00004.png,The scene contains a small peach die showing f...,3,4 3 5,peach yellow red,small large small
2,sample_00007.png,This picture contains a medium green die showi...,3,5 3 6,green purple purple,medium small small


## 3. Symbol Field Overview

Quick look at the distribution of the ground truth symbol fields.

In [3]:
print("=== Symbol field distributions (train set) ===")
print("\nnum_dice distribution:")
print(train_df["sym_num_dice"].value_counts().sort_index())

# Flatten space-separated colour and size fields to get per-die distributions
all_colours = [c for row in train_df["sym_colours"] for c in row.split()]
all_sizes   = [s for row in train_df["sym_sizes"]   for s in row.split()]
all_values  = [int(v) for row in train_df["sym_values"] for v in row.split()]

print("\nColour distribution (per die):")
colour_series = pd.Series(all_colours).value_counts()
print(colour_series)

print("\nSize distribution (per die):")
size_series = pd.Series(all_sizes).value_counts().reindex(VALID_SIZES)
print(size_series)

print("\nFace value distribution (per die):")
value_series = pd.Series(all_values).value_counts().sort_index()
print(value_series)

# Validate no unexpected values
unexpected_colours = set(all_colours) - set(VALID_COLOURS)
unexpected_sizes   = set(all_sizes)   - set(VALID_SIZES)
assert not unexpected_colours, f"Unexpected colours found: {unexpected_colours}"
assert not unexpected_sizes,   f"Unexpected sizes found: {unexpected_sizes}"
print("\nValidation passed: all symbol values are within expected ranges.")

=== Symbol field distributions (train set) ===

num_dice distribution:
sym_num_dice
1    1384
2    1413
3    1384
Name: count, dtype: int64

Colour distribution (per die):
red       1222
blue      1210
yellow    1209
green     1193
peach     1189
white     1181
purple    1158
Name: count, dtype: int64

Size distribution (per die):
small     2743
medium    2899
large     2720
Name: count, dtype: int64

Face value distribution (per die):
1    1405
2    1394
3    1430
4    1399
5    1341
6    1393
Name: count, dtype: int64

Validation passed: all symbol values are within expected ranges.


## 4. One-Hot Encoding

Treats each unique sentence as a category. Included as a weak baseline only — the high UNK rate on val/test makes it unsuitable for generalisation.

> With the richer sentences (now including size), expect more unique sentences and a higher UNK rate than before.

In [4]:
def build_sentence_vocab(texts: list) -> tuple:
    """Build a sentence-to-id mapping from training texts only."""
    unique_texts  = sorted(set(texts))
    sentence_to_id = {text: i for i, text in enumerate(unique_texts)}
    unk_id = len(sentence_to_id)
    return sentence_to_id, unk_id


def one_hot_encode(text: str, sentence_to_id: dict, unk_id: int) -> np.ndarray:
    vocab_size = len(sentence_to_id) + 1
    vec = np.zeros(vocab_size, dtype=np.float32)
    vec[sentence_to_id.get(text, unk_id)] = 1.0
    return vec


def encode_split_onehot(texts: list, sentence_to_id: dict, unk_id: int) -> np.ndarray:
    return np.array([one_hot_encode(t, sentence_to_id, unk_id) for t in texts])


sentence_to_id, UNK_ID = build_sentence_vocab(train_texts)
logger.info(f"One-hot vocab: {len(sentence_to_id)} unique training sentences + 1 UNK.")

X_train_onehot = encode_split_onehot(train_texts, sentence_to_id, UNK_ID)
X_val_onehot   = encode_split_onehot(val_texts,   sentence_to_id, UNK_ID)
X_test_onehot  = encode_split_onehot(test_texts,  sentence_to_id, UNK_ID)

print("Train one-hot shape:", X_train_onehot.shape)
print("Val one-hot shape:  ", X_val_onehot.shape)
print("Test one-hot shape: ", X_test_onehot.shape)

assert np.all(X_train_onehot.sum(axis=1) == 1.0)
assert np.all(X_val_onehot.sum(axis=1)   == 1.0)
assert np.all(X_test_onehot.sum(axis=1)  == 1.0)
print("Sanity check passed: all rows sum to 1.0")

unk_val  = int(np.sum(np.argmax(X_val_onehot,  axis=1) == UNK_ID))
unk_test = int(np.sum(np.argmax(X_test_onehot, axis=1) == UNK_ID))
print(f"\nUNK in val:  {unk_val}/{len(val_texts)}   ({100*unk_val/len(val_texts):.1f}% unseen)")
print(f"UNK in test: {unk_test}/{len(test_texts)} ({100*unk_test/len(test_texts):.1f}% unseen)")
print(f"\nExample: '{train_texts[0]}'")
print(f"One-hot: {X_train_onehot[0]}")

INFO: One-hot vocab: 3257 unique training sentences + 1 UNK.


Train one-hot shape: (4181, 3258)
Val one-hot shape:   (899, 3258)
Test one-hot shape:  (898, 3258)
Sanity check passed: all rows sum to 1.0

UNK in val:  601/899   (66.9% unseen)
UNK in test: 607/898 (67.6% unseen)

Example: 'The image shows a small purple die showing two, a small green die showing one and a medium blue die showing two.'
One-hot: [0. 0. 0. ... 0. 0. 0.]


## 5. TF-IDF

Word + bigram TF-IDF, fitted on training data only. Now that sentences include size words (small / medium / large), the vocabulary and feature space will be richer than before.

In [5]:
tfidf_vectorizer = TfidfVectorizer(
    ngram_range=(1, 2),  # unigrams + bigrams (captures e.g. 'large blue', 'showing three')
    min_df=2,            # drop terms appearing in fewer than 2 documents
    sublinear_tf=True,   # log(1 + tf) dampens high raw frequencies
)

# Fit on training data only
X_train_tfidf = tfidf_vectorizer.fit_transform(train_texts)
X_val_tfidf   = tfidf_vectorizer.transform(val_texts)
X_test_tfidf  = tfidf_vectorizer.transform(test_texts)

print("Train TF-IDF shape:", X_train_tfidf.shape)
print("Val TF-IDF shape:  ", X_val_tfidf.shape)
print("Test TF-IDF shape: ", X_test_tfidf.shape)
print(f"Vocabulary size:    {len(tfidf_vectorizer.vocabulary_)} terms")

assert X_train_tfidf.shape[1] == X_val_tfidf.shape[1] == X_test_tfidf.shape[1]
print("Sanity check passed: feature dims match across splits.")

# Show top features for first example, sorted by weight
feature_names   = tfidf_vectorizer.get_feature_names_out()
example_vector  = X_train_tfidf[0].toarray()[0]
nonzero_indices = example_vector.nonzero()[0]

print(f"\nExample: '{train_texts[0]}'")
print("Non-zero TF-IDF values (sorted by weight):")
for idx in sorted(nonzero_indices, key=lambda i: -example_vector[i]):
    print(f"  {feature_names[idx]:<25} -> {example_vector[idx]:.4f}")

Train TF-IDF shape: (4181, 178)
Val TF-IDF shape:   (899, 178)
Test TF-IDF shape:  (898, 178)
Vocabulary size:    178 terms
Sanity check passed: feature dims match across splits.

Example: 'The image shows a small purple die showing two, a small green die showing one and a medium blue die showing two.'
Non-zero TF-IDF values (sorted by weight):
  two small                 -> 0.3516
  showing two               -> 0.2705
  shows small               -> 0.2567
  two                       -> 0.2511
  small purple              -> 0.2367
  small green               -> 0.2287
  medium blue               -> 0.2238
  one and                   -> 0.2098
  small                     -> 0.1861
  die showing               -> 0.1833
  showing                   -> 0.1826
  and medium                -> 0.1667
  image                     -> 0.1598
  image shows               -> 0.1598
  shows                     -> 0.1598
  the image                 -> 0.1598
  purple die                -> 0.1592
  purpl

## 6. Sentence-BERT (SBERT)

Dense 384-dimensional semantic embeddings. Handles all unseen sentences and captures meaning — semantically similar sentences (e.g. same dice, different template) will have similar vectors.

In [ ]:
try:
    sbert_model = SentenceTransformer("all-MiniLM-L6-v2")
except Exception as e:
    raise RuntimeError(
        f"Failed to load SentenceTransformer: {e}\n"
        "Install with: pip install sentence-transformers"
    ) from e

X_train_sbert = sbert_model.encode(train_texts, convert_to_numpy=True, show_progress_bar=True, batch_size=64)
X_val_sbert   = sbert_model.encode(val_texts,   convert_to_numpy=True, show_progress_bar=True, batch_size=64)
X_test_sbert  = sbert_model.encode(test_texts,  convert_to_numpy=True, show_progress_bar=True, batch_size=64)

print("Train SBERT shape:", X_train_sbert.shape)
print("Val SBERT shape:  ", X_val_sbert.shape)
print("Test SBERT shape: ", X_test_sbert.shape)

assert X_train_sbert.shape[1] == 384
assert X_train_sbert.shape[0] == len(train_texts)
assert not np.isnan(X_train_sbert).any(), "NaN values in SBERT embeddings"
print("\nSanity checks passed.")

# Quick semantic similarity check — sentences about the same symbol should be close
from numpy.linalg import norm
def cosine_sim(a, b):
    return np.dot(a, b) / (norm(a) * norm(b))

sim_same  = cosine_sim(X_train_sbert[0], X_train_sbert[1])
sim_diff  = cosine_sim(X_train_sbert[0], X_train_sbert[-1])
print(f"\nExample similarity (consecutive sentences): {sim_same:.3f}")
print(f"Example similarity (distant sentences):     {sim_diff:.3f}")
print(f"\nExample: '{train_texts[0]}'")
print(f"SBERT (first 10): {X_train_sbert[0][:10]}")

INFO: No device provided, using mps
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/modules.json "HTTP/1.1 200 OK"
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/c9745ed1d9f207416be6d2e6f8de32d1f16199bf/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO: Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO: HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO: H

## 7. Summary

Comparison of all three representations. These will be used as prediction targets in A3.

In [ ]:
summary = pd.DataFrame([
    {
        "Method":           "One-Hot",
        "Train shape":      X_train_onehot.shape,
        "Feature dim":      X_train_onehot.shape[1],
        "Handles unseen?":  f"No — UNK slot ({unk_val} val, {unk_test} test)",
        "Dense/Sparse":     "Dense",
        "Captures size?":   "Yes (sentence contains size word)",
    },
    {
        "Method":           "TF-IDF",
        "Train shape":      X_train_tfidf.shape,
        "Feature dim":      X_train_tfidf.shape[1],
        "Handles unseen?":  "Partial (known words only)",
        "Dense/Sparse":     "Sparse",
        "Captures size?":   "Yes (size words in vocabulary)",
    },
    {
        "Method":           "SBERT",
        "Train shape":      X_train_sbert.shape,
        "Feature dim":      X_train_sbert.shape[1],
        "Handles unseen?":  "Yes (semantic generalisation)",
        "Dense/Sparse":     "Dense",
        "Captures size?":   "Yes (semantically encoded)",
    },
])

print(summary.to_string(index=False))

print("\n--- Available arrays for A3 ---")
print("One-hot : X_train_onehot, X_val_onehot,  X_test_onehot")
print("TF-IDF  : X_train_tfidf,  X_val_tfidf,   X_test_tfidf")
print("SBERT   : X_train_sbert,  X_val_sbert,   X_test_sbert")
print("\nSymbol ground truth available in train_df / val_df / test_df:")
print("  sym_num_dice, sym_values, sym_colours, sym_sizes")